# Notebook 03 — Statcast Data Pull (Baseball Savant)

Source: `baseballsavant.mlb.com` custom leaderboard CSV export  
Rate limit: 1 s between calls, browser-like User-Agent  

This notebook pulls two Statcast leaderboards for the 2025 season:

1. Hitting — xBA, xSLG, xwOBA, xOBP, xISO, avg_swing_speed, fast_swing_rate, blasts_contact, blasts_swing, squared_up_contact, squared_up_swing, avg_swing_length, swords, attack_angle, attack_direction, ideal_angle_rate, vertical_swing_path, exit_velocity_avg  launch_angle_avg, sweet_spot_percent, barrel_batted_rate
2. Pitching — xERA, xBA, xSLG, xwOBA, xOBP, xISO, wOBA, k_percent, bb_percent, whiff_percent, chase_percent, exit_velocity_avg  sweet_spot_percent, barrel_batted_rate, hard_hit_percent, groundballs_percent  

Join key is `mlbam_id` (the `player_id` column in Savant's CSV).

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from scrapers.baseball_savant import get_statcast_hitting, get_statcast_pitching
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## 1. Statcast Hitting Leaderboard
### What each metric measures and why it's more predictive

| Statcast Metric | Group | What it is |
|---|---|---|
| **xBA** (Expected Batting Average) | Expected Outcomes | Built from exit velocity and launch angle on every batted ball. Removes BABIP noise from batting average. Year-to-year correlation is ~0.75 vs. ~0.60 for AVG. |
| **xSLG** (Expected Slugging) | Expected Outcomes | Same EV/LA model applied to extra-base power. Reflects true contact quality regardless of park effects or whether a fly ball carried. |
| **xwOBA** (Expected Weighted On-Base Average) | Expected Outcomes | Combines contact quality with walk/strikeout rates on a run-value scale. More stable than actual wOBA because defense and park are removed from batted-ball outcomes. |
| **xOBP** (Expected On-Base Percentage) | Expected Outcomes | Applies the expected model to hit outcomes while keeping real BB/HBP. Removes BABIP luck from the hits component. |
| **xISO** (Expected Isolated Power) | Expected Outcomes | Expected extra-base power per at-bat. Mirrors traditional ISO but uses the EV/LA model to separate true power from park and luck. |
| **Avg Swing Speed** | Bat Tracking | Average bat speed measured 6 inches from the barrel across competitive swings. Faster swings create more potential exit velocity. |
| **Fast Swing Rate** | Bat Tracking | Percentage of swings at or above 75 mph bat speed. Shows how often a hitter generates elite swing speed. |
| **Blasts (Contact %)** | Bat Tracking | Percentage of contact events classified as a "blast": a swing that is both fast and squared up. Strong single-metric for hard contact production. |
| **Blasts (Swing %)** | Bat Tracking | Percentage of all swings that produce blast contact. Lower than blasts_contact because it includes swings that miss entirely. |
| **Squared-Up Contact %** | Bat Tracking | Percentage of contact where the batter reached close to the maximum possible exit velocity given bat speed and pitch speed. Tracks contact precision. |
| **Squared-Up Swing %** | Bat Tracking | Percentage of all swings (including misses) that result in squared-up contact. Paired with whiff rate to gauge overall swing efficiency. |
| **Avg Swing Length** | Bat Tracking | Average distance the barrel travels from start to contact in XYZ space (feet). Shorter paths mean quicker swings and less vulnerability to inside pitches. |
| **Swords** | Bat Tracking | Count of swings resulting in an extreme miss. The batter is beaten badly enough that the swing looks awkward. High sword counts signal vulnerability to quality stuff. |
| **Attack Angle** | Bat Tracking | Vertical angle of the bat at contact (degrees). Positive = upward swing plane. Optimal range (~5–15°) aligns with average pitch descent to maximize contact area. |
| **Attack Direction** | Bat Tracking | Horizontal swing direction at contact relative to home plate. Reflects pull tendency vs. opposite-field approach in the swing mechanics, independent of batted-ball outcomes. |
| **Ideal Angle Rate** | Bat Tracking | Percentage of swings where attack angle falls in the optimal launch window. Higher rates mean the swing plane consistently matches pitch trajectory. |
| **Vertical Swing Path** | Bat Tracking | Steepness of the swing plane from start to contact. Works alongside attack angle to describe the full 3D geometry of the swing. |
| **Exit Velocity (Avg)** | Batted Ball Outcomes | Average speed off the bat across all batted balls. Direct product of swing speed and contact quality. Strong predictor of wOBA and HR rate. |
| **Launch Angle (Avg)** | Batted Ball Outcomes | Average vertical angle of the batted ball. The 10–30° range produces the most run value. Explains why two hitters with the same EV can have very different results. |
| **Sweet Spot %** | Batted Ball Outcomes | Percentage of batted balls hit between 8° and 32° launch angle. Balls in this window have the highest hit probability and slugging potential. |
| **Barrel %** | Batted Ball Outcomes | Percentage of batted balls meeting the optimal EV/LA threshold (≥98 mph, 26–30°). Barrels carry a ~1.000 xSLG and are the most consistent power indicator year over year.|

In [4]:
hitting = get_statcast_hitting(2025)
print(f"Shape: {hitting.shape}")
print(f"\nDtypes:\n{hitting.dtypes}")
print(f"\nNull rates:\n{hitting.isnull().mean().round(4)}")
print(f"\nSample (10 rows):")
hitting.head(10)

Shape: (537, 23)

Dtypes:
mlbam_id                 int64
name                    object
xBA                    float64
xSLG                   float64
xwOBA                  float64
xOBP                   float64
xISO                   float64
avg_swing_speed        float64
fast_swing_rate        float64
blasts_contact         float64
blasts_swing           float64
squared_up_contact     float64
squared_up_swing       float64
avg_swing_length       float64
swords                   int64
attack_angle           float64
attack_direction       float64
ideal_angle_rate       float64
vertical_swing_path    float64
exit_velocity_avg      float64
launch_angle_avg       float64
sweet_spot_percent     float64
barrel_batted_rate     float64
dtype: object

Null rates:
mlbam_id               0.0
name                   0.0
xBA                    0.0
xSLG                   0.0
xwOBA                  0.0
xOBP                   0.0
xISO                   0.0
avg_swing_speed        0.0
fast_swing_rate   

,mlbam_id,name,xBA,xSLG,xwOBA,xOBP,xISO,avg_swing_speed,fast_swing_rate,blasts_contact,blasts_swing,squared_up_contact,squared_up_swing,avg_swing_length,swords,attack_angle,attack_direction,ideal_angle_rate,vertical_swing_path,exit_velocity_avg,launch_angle_avg,sweet_spot_percent,barrel_batted_rate
0,683146,"Baty, Brett",0.252,0.457,0.334,0.313,0.205,74.8,48.5,17.5,12.8,30.9,22.7,7.4,16,8.4,3.7,47.6,35.5,90.7,5.8,32.6,12.8
1,687515,"Thomas, Colby",0.206,0.383,0.281,0.256,0.176,74.8,49.2,19.8,13.2,27.2,18.2,7.7,7,11.6,-5.0,55.8,28.9,92.2,21.2,24.0,13.3
2,682729,"Clase, Jonatan",0.217,0.341,0.284,0.295,0.124,71.8,21.9,12.1,8.9,28.2,20.7,6.7,4,7.0,-1.8,46.2,32.3,86.0,17.4,21.9,6.8
3,647351,"Toro, Abraham",0.253,0.376,0.302,0.309,0.123,69.6,6.0,11.3,9.1,32.9,26.6,7.6,6,11.2,-2.8,52.9,32.0,86.3,13.4,32.0,4.5
4,666023,"Fermin, Freddy",0.233,0.336,0.271,0.280,0.104,69.6,5.3,10.7,8.5,33.1,26.3,7.3,21,8.9,-0.7,49.5,34.5,89.1,14.2,28.7,4.2
5,663616,"Larnach, Trevor",0.247,0.399,0.315,0.321,0.153,72.2,23.2,16.7,12.0,37.9,27.1,7.3,19,8.3,-10.6,41.1,34.1,90.9,10.9,34.9,7.2
6,663837,"Vierling, Matt",0.250,0.436,0.338,0.326,0.186,71.7,11.0,15.6,13.4,32.7,27.9,7.4,2,6.8,1.7,40.7,31.6,92.2,20.4,27.3,12.1
7,663886,"Stephenson, Tyler",0.216,0.413,0.313,0.306,0.197,70.1,4.9,14.2,10.3,31.6,22.9,7.4,26,13.3,0.7,51.3,37.5,90.5,12.9,42.2,14.4
8,691016,"Soderstrom, Tyler",0.261,0.457,0.341,0.333,0.196,74.0,37.9,20.8,15.9,33.0,25.2,7.5,27,9.6,1.2,50.2,30.6,91.6,7.9,34.4,11.4
9,678894,"Peguero, Liover",0.177,0.295,0.246,0.251,0.119,73.1,22.2,12.9,8.7,22.4,15.1,7.5,1,6.9,-9.9,42.1,25.0,85.7,6.5,21.8,9.1


## 2. Statcast Pitching Leaderboard
### What each metric measures and why it's more predictive

| Statcast Metric | Group | What it is |
|---|---|---|
| **xERA** (Expected ERA) | Expected Outcomes | ERA built from contact quality allowed (EV + LA model) rather than actual runs. Removes defense, sequencing, and strand-rate luck, making it more park- and defense-neutral than standard ERA estimators. |
| **xBA** (Expected Batting Average Allowed) | Expected Outcomes | Expected hit rate against, based on the EV/LA of every batted ball the pitcher allowed. Removes BABIP noise. A pitcher with low xBA but high BA is likely benefiting from good defense or sequencing. |
| **xSLG** (Expected Slugging Allowed) | Expected Outcomes | EV/LA model applied to extra-base contact allowed. Reflects true power damage regardless of park or whether fly balls happened to carry. |
| **xwOBA** (Expected wOBA Allowed) | Expected Outcomes | Combines contact quality with real strikeouts and walks on a run-value scale. More stable than actual wOBA allowed because defense is removed from batted-ball outcomes. Widely considered the strongest single contact-quality read on a pitcher. |
| **xOBP** (Expected OBP Allowed) | Expected Outcomes | Applies the expected model to hit outcomes while keeping real BB/HBP allowed. Distinguishes between high OBP allowed driven by walks versus hit luck. |
| **xISO** (Expected ISO Allowed) | Expected Outcomes | Expected extra-base power allowed per at-bat. Separates true power suppression from park effects and HR/FB luck. |
| **wOBA** (Weighted On-Base Average Allowed) | Expected Outcomes | Actual run-value-weighted on-base average allowed. Included alongside xwOBA so the gap between the two can be tracked as a luck indicator. |
| **K%** (Strikeout Rate) | Plate Discipline | Strikeouts per plate appearance. Defense plays no role in strikeouts, which is why K% holds up better than most pitching stats year over year and is a strong predictor of ERA estimators. |
| **BB%** (Walk Rate) | Plate Discipline | Walks per plate appearance. Free baserunners with no defense involved. Combined with K% into K-BB%, it gives the clearest read on a pitcher's command. |
| **Whiff %** | Plate Discipline | Swings and misses divided by total swings. Tracks swing-and-miss ability on contact attempts specifically, making it more granular than SwStr%, which includes pitches batters never offered at. |
| **Chase %** (OZ Swing %) | Plate Discipline | Percentage of pitches outside the zone that batters swing at. Higher rates mean hitters are making poor decisions, usually a function of pitch quality and location at the edges. |
| **Exit Velocity (Avg Allowed)** | Batted Ball Outcomes | Average EV on all batted balls against. Shows how hard the pitcher is being hit. Less park-dependent than ERA or HR rate. |
| **Sweet Spot % (Allowed)** | Batted Ball Outcomes | Percentage of batted balls against landing in the 8–32° launch angle window. Balls in this range have the highest hit probability and slugging potential. Keeping this rate low means the pitcher is generating weak or mis-hit contact. |
| **Barrel % (Allowed)** | Batted Ball Outcomes | Percentage of batted balls against meeting the barrel threshold (≥98 mph, 26–30° LA). Barrels carry ~1.000 xSLG against and this rate is one of the more consistent damage-allowed metrics year over year. |
| **Hard-Hit % (Allowed)** | Batted Ball Outcomes | Percentage of batted balls against at ≥95 mph exit velocity. Broader than barrel rate, it picks up overall contact damage including line drives that don't meet the barrel angle threshold. |
| **GB% (Allowed)** | Batted Ball Outcomes | Ground ball rate on batted balls against. Ground balls have the lowest xSLG of any batted ball type and can't leave the park. High GB% pitchers are more defense-dependent but limit big innings. |

In [6]:
pitching = get_statcast_pitching(2025)
print(f"Shape: {pitching.shape}")
print(f"\nDtypes:\n{pitching.dtypes}")
print(f"\nNull rates:\n{pitching.isnull().mean().round(4)}")
print(f"\nSample (10 rows):")
pitching.head(10)

Shape: (657, 18)

Dtypes:
mlbam_id                 int64
name                    object
xERA                   float64
xBA                    float64
xSLG                   float64
xwOBA                  float64
xOBP                   float64
xISO                   float64
wOBA                   float64
k_percent              float64
bb_percent             float64
whiff_percent          float64
chase_percent          float64
exit_velocity_avg      float64
sweet_spot_percent     float64
barrel_batted_rate     float64
hard_hit_percent       float64
groundballs_percent    float64
dtype: object

Null rates:
mlbam_id               0.0
name                   0.0
xERA                   0.0
xBA                    0.0
xSLG                   0.0
xwOBA                  0.0
xOBP                   0.0
xISO                   0.0
wOBA                   0.0
k_percent              0.0
bb_percent             0.0
whiff_percent          0.0
chase_percent          0.0
exit_velocity_avg      0.0
sweet_spot_

,mlbam_id,name,xERA,xBA,xSLG,xwOBA,xOBP,xISO,wOBA,k_percent,bb_percent,whiff_percent,chase_percent,exit_velocity_avg,sweet_spot_percent,barrel_batted_rate,hard_hit_percent,groundballs_percent
0,663969,"Phillips, Tyler",3.66,0.240,0.370,0.298,0.303,0.131,0.284,16.6,7.7,27.3,34.7,89.2,26.0,5.1,43.4,55.7
1,680694,"Bradish, Kyle",3.09,0.210,0.341,0.275,0.279,0.131,0.255,37.3,7.9,34.8,31.2,89.8,45.6,8.8,38.2,33.8
2,608032,"Estévez, Carlos",3.69,0.228,0.404,0.299,0.302,0.176,0.255,20.1,8.2,19.0,23.0,89.1,35.4,10.6,38.1,25.4
3,640448,"Finnegan, Kyle",3.16,0.226,0.338,0.278,0.291,0.112,0.273,24.0,7.9,24.6,27.7,90.4,32.5,7.8,42.9,49.4
4,676604,"Zuber, Tyler",6.64,0.298,0.563,0.387,0.365,0.266,0.416,23.7,10.2,19.8,32.7,88.3,53.8,15.4,41.0,28.2
5,700241,"McGreevy, Michael",4.67,0.274,0.457,0.333,0.316,0.183,0.318,14.5,5.0,19.2,25.3,90.5,33.2,8.8,40.1,48.0
6,666142,"Ragans, Cole",2.67,0.187,0.321,0.256,0.256,0.135,0.295,38.1,7.8,34.8,29.8,88.4,32.1,9.5,39.4,37.2
7,675848,"Mejia, Juan",2.98,0.199,0.297,0.270,0.295,0.098,0.309,26.1,9.6,30.4,30.4,85.8,34.0,4.9,34.6,38.9
8,650893,"Cabrera, Génesis",4.67,0.246,0.452,0.333,0.324,0.206,0.380,18.7,9.6,27.1,28.6,89.5,32.6,11.4,42.4,39.4
9,665152,"Kremer, Dean",3.82,0.238,0.413,0.304,0.289,0.175,0.305,20.1,6.4,22.9,30.1,88.1,32.8,8.5,35.8,40.2


## 3. Null Rate Summary (both DataFrames)

In [8]:
null_summary = pd.DataFrame({
    "hitting": hitting.isnull().mean().round(4),
    "pitching": pitching.isnull().mean().round(4),
}).fillna("—")
print("=== Null Rate Summary ===")
null_summary

=== Null Rate Summary ===


,hitting,pitching
attack_angle,0.0,—
attack_direction,0.0,—
avg_swing_length,0.0,—
avg_swing_speed,0.0,—
barrel_batted_rate,0.0,0.0
bb_percent,—,0.0
blasts_contact,0.0,—
blasts_swing,0.0,—
chase_percent,—,0.0
exit_velocity_avg,0.0,0.0


## 4. Save to Parquet


In [10]:
data_dir = os.path.join("..", "data")
os.makedirs(data_dir, exist_ok=True)

hitting.to_parquet(os.path.join(data_dir, "raw_statcast_hitting.parquet"), index=False)
pitching.to_parquet(os.path.join(data_dir, "raw_statcast_pitching.parquet"), index=False)

print("Saved:")
for f in ["raw_statcast_hitting.parquet", "raw_statcast_pitching.parquet"]:
    path = os.path.join(data_dir, f)
    size_kb = os.path.getsize(path) / 1024
    print(f"  {f} — {size_kb:.1f} KB")

Saved:
  raw_statcast_hitting.parquet — 54.1 KB
  raw_statcast_pitching.parquet — 49.4 KB
